# 3.2 — Model Refinement (Top-15 Features)

Notebook ini melakukan penyempurnaan model dengan melatih ulang model hanya menggunakan 15 fitur terbaik berdasarkan feature importance dari tahap 3.0.

**Input:**
- `2_data_preprocessing/output/2.2_final_feature_set.csv`
- `3_modelling/output/top_15_feature_importance.csv`

**Output:**
- `3_modelling/output/model_metrics_top15.csv`
- `3_modelling/output/3_model_predictions_top15.csv`

In [6]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Gunakan XGBoost jika tersedia; fallback ke RandomForest agar notebook tetap runnable.
try:
    import xgboost as xgb
    use_xgboost = True
    model_name = 'XGBoost (Top-15)'
except ImportError:
    from sklearn.ensemble import RandomForestRegressor
    use_xgboost = False
    model_name = 'RandomForest (Top-15)'

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / '2_data_preprocessing' / 'output' / '2.2_final_feature_set.csv').exists():
            return p
    raise FileNotFoundError('Could not find 2.2_final_feature_set.csv in current or parent directories.')

ROOT = find_project_root(Path.cwd())
input_path = ROOT / '2_data_preprocessing' / 'output' / '2.2_final_feature_set.csv'
output_dir = ROOT / '3_modelling' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)

top15_path = output_dir / 'top_15_feature_importance.csv'
metrics_path = output_dir / 'model_metrics_top15.csv'
pred_path = output_dir / '3_model_predictions_top15.csv'

df = pd.read_csv(input_path)
df['tanggal'] = pd.to_datetime(df['tanggal'])
df = df.sort_values(by=['provinsi_id', 'tanggal']).reset_index(drop=True)

print(f'Loaded: {input_path}')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} cols')
print(f'Read top-15 features from: {top15_path}')

Loaded: C:\Users\Unpad-hci\Microsoft-Datathon-TWP90-Early-Warning-System\2_data_preprocessing\output\2.2_final_feature_set.csv
Shape: 1,488 rows × 36 cols
Read top-15 features from: C:\Users\Unpad-hci\Microsoft-Datathon-TWP90-Early-Warning-System\3_modelling\output\top_15_feature_importance.csv


In [7]:
# Siapkan data modelling + ambil 15 fitur terbaik
lag_columns = [col for col in df.columns if '_lag_' in col]
required_cols = lag_columns + ['twp90_pct']
df_model = df.dropna(subset=required_cols).copy()

df_model['tahun'] = pd.to_numeric(df_model['tahun'], errors='coerce')
df_model = df_model.dropna(subset=['tahun'])
df_model['tahun'] = df_model['tahun'].astype(int)

kolom_non_prediktor = ['provinsi_id', 'nama_provinsi', 'tanggal', 'tahun', 'bulan', 'twp90_pct']
all_feature_cols = [col for col in df_model.columns if col not in kolom_non_prediktor]

if not top15_path.exists():
    raise FileNotFoundError(
        f'Top-15 feature file not found: {top15_path}. Jalankan dulu notebook 3.0 untuk membuatnya.'
    )

top15_df = pd.read_csv(top15_path)
if 'feature' not in top15_df.columns:
    raise ValueError(f'CSV {top15_path} harus memiliki kolom: feature')

top_features = top15_df['feature'].astype(str).tolist()
# Deduplicate (tetap menjaga urutan)
top_features = list(dict.fromkeys(top_features))

missing = [f for f in top_features if f not in df_model.columns]
if missing:
    print('Warning: beberapa fitur top-15 tidak ada di dataset, akan di-skip:')
    print(missing)
    top_features = [f for f in top_features if f in df_model.columns]

if len(top_features) == 0:
    raise ValueError('Tidak ada fitur top-15 yang cocok dengan kolom dataset.')

X_cols = top_features
print(f'Total rows available for modelling: {len(df_model):,}')
print(f'Total features available: {len(all_feature_cols)}')
print(f'Features used (Top-{len(X_cols)}): {X_cols}')

Total rows available for modelling: 1,302
Total features available: 30
Features used (Top-15): ['x7_jumlah_kc_bank_lag_6', 'x6_tabungan_miliar', 'x3_pdrb_per_kapita', 'x7_jumlah_kc_bank_lag_3', 'x10_rasio_umkm_lag_3', 'x9_npl_ratio', 'x1_bi_rate_pct', 'x6_tabungan_miliar_lag_3', 'x7_jumlah_kc_bank', 'x4_tpt_pct_lag_3', 'x5_penetrasi_internet_pct_lag_3', 'x3_pdrb_per_kapita_lag_3', 'x4_tpt_pct_lag_6', 'x6_tabungan_miliar_lag_6', 'x5_penetrasi_internet_pct']


In [8]:
# Split temporal: Train 2022-2024 | Test 2025 (fallback jika kosong)
train_data = df_model[df_model['tahun'] <= 2024].copy()
test_data = df_model[df_model['tahun'] == 2025].copy()

if train_data.empty or test_data.empty:
    print('Warning: strict temporal split produced an empty set. Using 80/20 chronological fallback split.')
    df_model = df_model.sort_values('tanggal').reset_index(drop=True)
    split_idx = int(0.8 * len(df_model))
    train_data = df_model.iloc[:split_idx].copy()
    test_data = df_model.iloc[split_idx:].copy()

X_train = train_data[X_cols]
y_train = train_data['twp90_pct']
X_test = test_data[X_cols]
y_test = test_data['twp90_pct']

print(f'Dimensi X_train: {X_train.shape}')
print(f'Dimensi X_test: {X_test.shape}')

assert not X_train.empty and not X_test.empty, 'Dataset train/test kosong setelah preprocessing.'
assert (df_model['provinsi_id'] == 19).sum() == 0, 'provinsi_id 19 still present (expected removed upstream).'

Dimensi X_train: (930, 15)
Dimensi X_test: (372, 15)


In [9]:
# Latih model refined dengan Top-15 features + evaluasi
if use_xgboost:
    model = xgb.XGBRegressor(
        objective='reg:squarederror',
        n_estimators=1200,
        learning_rate=0.03,
        max_depth=5,
        subsample=0.85,
        colsample_bytree=0.85,
        eval_metric='rmse',
        early_stopping_rounds=50,
        random_state=42,
    )

    # Early stopping pakai split validasi dari data training (berdasarkan waktu)
    train_sorted = train_data.sort_values('tanggal').reset_index(drop=True)
    val_size = int(0.2 * len(train_sorted))

    if val_size >= 10:
        train_part = train_sorted.iloc[:-val_size].copy()
        val_part = train_sorted.iloc[-val_size:].copy()
        X_tr = train_part[X_cols]
        y_tr = train_part['twp90_pct']
        X_val = val_part[X_cols]
        y_val = val_part['twp90_pct']
        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False,
        )
    else:
        model.fit(X_train, y_train)
else:
    model = RandomForestRegressor(
        n_estimators=500,
        random_state=42,
        n_jobs=-1,
        max_features='sqrt',
    )
    model.fit(X_train, y_train)

y_pred = model.predict(X_test)

rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
mae = float(mean_absolute_error(y_test, y_pred))
r2 = float(r2_score(y_test, y_pred))

print(f'--- Evaluasi Refined Model ({model_name}) ---')
print(f'RMSE: {rmse:.4f}')
print(f'MAE: {mae:.4f}')
print(f'R2: {r2:.4f}')

metrics_df = pd.DataFrame([{
    'model': model_name,
    'rmse': rmse,
    'mae': mae,
    'r2': r2,
    'n_train': int(len(X_train)),
    'n_test': int(len(X_test)),
    'n_features': int(len(X_cols)),
}])
metrics_df.to_csv(metrics_path, index=False)
print(f'Saved: {metrics_path}')

out_cols = [c for c in ['provinsi_id', 'nama_provinsi', 'tanggal', 'tahun', 'bulan', 'twp90_pct'] if c in test_data.columns]
pred_df = test_data[out_cols].copy()
pred_df = pred_df.rename(columns={'twp90_pct': 'y_true'})
pred_df['y_pred'] = y_pred
pred_df['error'] = pred_df['y_pred'] - pred_df['y_true']
pred_df.to_csv(pred_path, index=False)
print(f'Saved: {pred_path}')

--- Evaluasi Refined Model (XGBoost (Top-15)) ---
RMSE: 0.0082
MAE: 0.0046
R2: 0.4193
Saved: C:\Users\Unpad-hci\Microsoft-Datathon-TWP90-Early-Warning-System\3_modelling\output\model_metrics_top15.csv
Saved: C:\Users\Unpad-hci\Microsoft-Datathon-TWP90-Early-Warning-System\3_modelling\output\3_model_predictions_top15.csv
